# **8. 데이터 분석을 위한 pandas 2**

## **학습 목표**

- `loc`·`iloc`·조건식·`query()`로 필요한 행과 열을 고르고, 조건에 맞는 값을 바꾼다.


---

# **0. 라이브러리 불러오기**


In [ ]:
import os            # 파일 경로를 다루는 표준 라이브러리
import numpy as np   # 배열 계산
import pandas as pd  # DataFrame 다루기

## **데이터 불러오기**

| 변수 | 파일 | 내용 |
|---|---|---|
| `df` | `sales.csv` | 주문 기록 15행. 개념 예제에 쓰는 작은 표 |
| `orders` | `superstore.xlsx`의 `Orders` 시트 | 주문상품 기록 10,194행. 한 행이 주문 한 건이 아니라 주문에 들어 있는 상품 하나다 |

CSV에서 읽은 날짜는 문자열이다. 날짜 범위로 조회하려면 `pd.to_datetime()`으로 날짜형으로 바꿔 둔다.

공식 문서: [pandas.read_csv()](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html) · [pandas.read_excel()](https://pandas.pydata.org/docs/reference/api/pandas.read_excel.html) · [pandas.to_datetime()](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html)


In [ ]:

from google.colab import drive

# Google Drive를 Colab에 연결한다. 실행하면 권한 승인 창이 나타난다 → "Google Drive에 연결" → 계정 선택 → "허용"
drive.mount("/content/drive")
data_folder = "/content/drive/MyDrive/HD-AI/data"

print("데이터 폴더:", data_folder)
print("data 폴더의 파일:", os.listdir(data_folder))
store_path = os.path.join(data_folder, "superstore.xlsx")

# 시트 이름 확인
print(pd.ExcelFile(store_path).sheet_names)       # ['Orders', 'People', 'Returns']

# 시트 이름으로 지정해 읽기
orders = pd.read_excel(store_path, sheet_name="Orders")
returns = pd.read_excel(store_path, sheet_name="Returns")



print("주문상품 기록 데이터 크기:", orders.shape)
print("반품 데이터 크기:", returns.shape)
returns.head()

csv_path = os.path.join(data_folder, "sales.csv")

df_csv = pd.read_csv(csv_path)

print("CSV 데이터 크기:", df_csv.shape)

df = df_csv.copy()
df["주문일"] = pd.to_datetime(df["주문일"])   # 문자열 → 날짜형 (날짜 범위 조회에 필요하다)
df.head()


# **1. 데이터 선택과 조건 조회**

필요한 부분만 골라내는 단계다. 열을 고르는 방법, 행을 고르는 방법, 조건으로 걸러내는 방법을 차례로 익힌다.

## **1) 단일 열 선택**

열 이름 하나를 대괄호에 넣으면 그 열만 꺼낼 수 있다.

```python
df["열이름"]
```

이때 결과는 열 하나짜리 1차원 데이터인 `Series`가 된다. 표가 아니라 값의 목록이라고 생각하면 된다. `Series`에도 `head()`, `mean()`, `sum()` 같은 기능을 그대로 사용할 수 있다.

공식 문서: [열 선택](https://pandas.pydata.org/docs/user_guide/indexing.html#basics)


In [ ]:
# [예제] 열 하나 고르기 → Series
sales_column = df["매출"]

print(type(sales_column))
sales_column.head()


<class 'pandas.core.series.Series'>


,매출
0,2400000
1,900000
2,300000
3,1200000
4,480000


### **[실습] 1. 단일 열 선택**


In [ ]:
# [실습] 단일 열 선택: Series 조회
# 1. df에서 '매출' 열 하나를 선택하여 sales_one에 저장하시오.
# 2. sales_one의 자료형(type)을 출력하시오.
# 3. sales_one의 처음 3개 값을 출력하시오.


## **2) 여러 열 선택**

열을 두 개 이상 고를 때는 열 이름을 리스트로 묶어 전달한다. 대괄호가 두 번 겹치는 이유는 안쪽 대괄호가 리스트를 뜻하기 때문이다.

```python
df[["열1", "열2"]]
```

```text
 df["매출"]            →  Series    1차원, 값의 목록
 df[["매출"]]          →  DataFrame 2차원, 열이 하나인 표
 df[["지역", "매출"]]   →  DataFrame 2차원, 열이 둘인 표
```

이름 하나만 넣으면 `Series`가 되고, 리스트로 넣으면 열이 하나여도 `DataFrame`이 된다. 표 형태를 유지하고 싶을 때는 리스트로 전달한다.


In [ ]:
# [예제] 열 여러 개 고르기 → DataFrame
selected_columns = df[["주문일", "지역", "제품군", "매출"]]

print(type(selected_columns))
selected_columns.head()


<class 'pandas.core.frame.DataFrame'>


,주문일,지역,제품군,매출
0,2026-01-05,서울,전자,2400000
1,2026-01-12,부산,가구,900000
2,2026-01-20,서울,식품,300000
3,2026-02-03,대전,전자,1200000
4,2026-02-14,부산,식품,480000


### **열 선택으로 Series와 DataFrame 구분하기**


- 대괄호에 열 이름 **하나**를 넣으면 `Series`가 나온다. `Series`는 값이 한 줄로 늘어선 1차원 데이터라서 `shape`에 숫자가 하나만 있다. 예: `(10194,)`
- 대괄호에 열 이름을 **리스트**로 넣으면 `DataFrame`이 나온다. `DataFrame`은 행과 열이 있는 2차원 표라서 `shape`에 (행 수, 열 수) 두 숫자가 있다.
- 열이 하나여도 리스트로 넣으면 `DataFrame`이다. `shape`을 보면 몇 차원인지 바로 알 수 있다.

공식 문서: [인덱싱 기본](https://pandas.pydata.org/docs/user_guide/indexing.html#basics) · [DataFrame.shape](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.shape.html)


In [ ]:
region = orders["Region"]
print(type(region))                 # Series

selected = orders[["Order ID", "Region", "Sales"]]
print(type(selected))               # DataFrame
print(selected.shape)               # (10194, 3)


print(orders["Sales"])
print(orders["Sales"].shape)        # (10194,)   1차원: 숫자가 하나


print(orders[["Sales"]])
print(orders[["Sales"]].shape)      # (10194, 1) 2차원: 열이 하나인 표

<class 'pandas.core.series.Series'>
<class 'pandas.core.frame.DataFrame'>
(10194, 3)
0         16.448
1          3.540
2         11.784
3        272.736
4         19.536
          ...   
10189     52.776
10190     20.720
10191      3.024
10192     90.930
10193      3.024
Name: Sales, Length: 10194, dtype: float64
(10194,)
         Sales
0       16.448
1        3.540
2       11.784
3      272.736
4       19.536
...        ...
10189   52.776
10190   20.720
10191    3.024
10192   90.930
10193    3.024

[10194 rows x 1 columns]
(10194, 1)


### **[실습] 2. 여러 열 선택**


In [ ]:
# [실습] 여러 열 선택: DataFrame 조회
# 1. df에서 '제품명', '매출' 열을 이 순서대로 선택하시오.
# 2. 선택한 결과의 처음 3행과 shape를 출력하시오.


## **3) loc**

`loc`는 **행 인덱스 레이블과 열 레이블(열 이름)** 을 기준으로 조회한다. 행 자리에 불리언 조건식도 사용할 수 있다. 인덱스를 직접 지정하지 않으면 기본적으로 0부터 시작하는 `RangeIndex`가 생성된다.

```python
df.loc[행, 열]
```

- 행 자리에는 인덱스 이름이나 조건식을 넣는다
- 열 자리에는 열 이름이나 열 이름 목록을 넣는다
- 열 자리를 생략하면 모든 열을 가져온다

`loc`에서 레이블 슬라이싱을 지정하면 **끝 레이블이 포함된다(해당 레이블이 존재하는 경우).** `df.loc[0:3]`은 레이블 0, 1, 2, 3인 행을 모두 가져오므로 총 4개 행이 조회된다. DAY 1에서 배운 리스트 슬라이싱과 다른 점이므로 주의한다.

공식 문서: [loc](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.loc.html)


In [ ]:
# [예제 1] loc: 전체 행에서 열 이름으로 선택
# 열 이름 하나: Series / 열 이름 목록: DataFrame
display(df.loc[:, "매출"].head(3))
display(df.loc[:, ["지역", "매출"]].head(3))


In [ ]:
# [예제 2] loc: 연속된 행 레이블과 필요한 열 선택
# 0~3번 레이블은 모두 포함한다.
display(df.loc[0:3, ["지역", "제품군", "매출"]])


In [ ]:
# [예제 3] loc: 떨어진 행 레이블과 필요한 열 선택
display(df.loc[[1, 3, 5], ["지역", "매출"]])


In [ ]:
# [예제 4] loc: 한 행을 레이블로 조회
display(df.loc[2])


In [ ]:
# [예제 5] loc: 한 행과 한 열의 값 조회
print(df.loc[2, "매출"])


### **[실습] 3. loc**


In [ ]:
# [실습] loc: 행 레이블과 열 이름으로 조회
# 1. 인덱스 레이블 1~3의 '지역', '매출' 열을 조회하시오.
# 2. 인덱스 레이블 2의 '제품명' 값을 하나 출력하시오.
# 힌트: loc 슬라이싱은 끝 레이블 3을 포함한다.


## **4) iloc**

`iloc`는 행과 열을 **0부터 시작하는 정수 위치**로 조회한다. 인덱스 레이블이 10, 20, 30처럼 달라도 위치는 0, 1, 2로 센다. 이름이 아니라 순서로 찾는다는 점이 `loc`와 다르다.

```python
df.iloc[행번호, 열번호]
```

`iloc`에서 범위를 지정하면 **끝 번호가 포함되지 않는다.** 파이썬에서 리스트를 자를 때와 같은 규칙이다. `df.iloc[0:4]`는 0, 1, 2, 3번 행만 가져온다.

반면 `loc`는 번호가 아니라 이름을 기준으로 찾기 때문에 끝 이름까지 포함한다. 지금 사용하는 데이터는 인덱스가 0부터 시작하는 숫자라 둘이 비슷해 보이지만, `loc`가 보는 `0`은 순서가 아니라 그 행의 이름이다.

### **loc와 iloc 비교**

| 구분 | `loc` | `iloc` |
|---|---|---|
| 기준 | 이름 또는 조건 | 위치 번호 |
| 열 지정 방법 | `["지역", "매출"]` | `1:3` 처럼 번호 |
| 끝 번호 | 포함한다 | 포함하지 않는다 |
| 예 | `df.loc[0:3]` → 4개 행 | `df.iloc[0:3]` → 3개 행 |

```python
# 인덱스 레이블이 1인 행 조회
df.loc[1]

# 위치가 1인 행 조회 (두 번째 행)
df.iloc[1]
```

공식 문서: [iloc](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.iloc.html)


In [ ]:
# [예제 1] iloc: 행과 열을 위치 범위로 조회
# 0~3번 행, 1~4번 열을 조회한다. 끝 위치는 포함되지 않는다.
display(df.iloc[0:4, 1:5])


### **iloc 위치별 예제**

`df.columns`를 먼저 확인하여 열의 위치를 구분한다.


In [ ]:
# [예제] iloc 조회
# 조회할 열의 이름과 순서를 확인한다.
print(list(enumerate(df.columns)))

[(0, '주문번호'), (1, '주문일'), (2, '지역'), (3, '제품군'), (4, '제품명'), (5, '수량'), (6, '매출')]


In [ ]:
# [예제 2] iloc: 한 행 / 한 열 / 한 값 조회
display(df.iloc[0])       # 첫 번째 행
display(df.iloc[:, 2])    # 세 번째 열
print(df.iloc[2, 2])      # 세 번째 행의 세 번째 열 값


In [ ]:
# [예제 3] iloc: 떨어진 행 위치 조회
display(df.iloc[[0, 2, 4]])


In [ ]:
# [예제 4] iloc: 떨어진 열 위치 조회
display(df.iloc[:, [0, 2]])


In [ ]:
# [예제 5] iloc: 연속된 행과 떨어진 열을 함께 조회
display(df.iloc[0:4, [0, 2]])


In [ ]:
# [예제 6] iloc: 2행 간격으로 조회
display(df.iloc[::2])


In [ ]:
# [예제] loc와 iloc: 같은 행을 서로 다른 방식으로 선택
mini = pd.DataFrame({"제품": ["A", "B", "C"], "수량": [10, 20, 30]}, index=["가", "나", "다"])
print(mini.loc["나", "수량"])    # 20: 라벨 기준
print(mini.iloc[1, 1])        # 20: 정수 위치 기준
display(mini.loc[["가", "다"]])  # 행 이름으로 여러 행 선택

20
20


,제품,수량
가,A,10
다,C,30


### **`iloc`으로 마지막 행·마지막 열 고르기**

`iloc`은 **음수 위치**를 사용할 수 있다. `-1`은 마지막 행 또는 마지막 열이고, `-3:`은 마지막 3행을 뜻한다.

```python
df.iloc[-1]       # 마지막 행
df.iloc[-3:]      # 마지막 3행
df.iloc[:, -1]    # 마지막 열
```

열 자리에도 같은 규칙을 쓴다. 자리에 `:`만 쓰면 **전체**를 뜻한다.

```python
df.iloc[:, :-1]   # 모든 행, 처음 열부터 마지막 열 직전까지 → 마지막 열만 빠진다
```

공식 문서: [iloc](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.iloc.html)


In [ ]:
# [예제 7] iloc: 음수 위치로 마지막 행·열 조회
display(df.iloc[-1])      # 마지막 행
display(df.iloc[-3:])     # 마지막 3행
display(df.iloc[:, -1])   # 마지막 열


In [ ]:
# [예제 8] iloc: 마지막 열만 빼고 고르기
# 마지막 열에 메모(비고)가 있는 센서 기록 예시
sensor = pd.DataFrame({"온도": [68, 71, 77, 74],
                       "진동": [0.4, 0.6, 1.1, 0.5],
                       "비고": ["정상", "정상", "점검", "정상"]})
values = sensor.iloc[:, :-1]   # 모든 행(:), 마지막 열 직전까지(:-1) → 온도·진동
memo = sensor.iloc[:, -1]      # 모든 행(:), 마지막 열 하나(-1) → 비고
display(values)
print(memo.tolist())

### **`iloc`으로 앞에서부터 몇 %만 고르기**

데이터를 앞부분과 뒷부분으로 나눌 때는 먼저 **몇 번째 행까지 고를지** 계산한다.

1. 전체 행 개수에 비율을 곱한다. 예: 15행의 70% → 15 × 0.7 = 10.5
2. `int()`로 소수점 이하를 버려 정수로 만든다. 10.5 → 10
3. `iloc[:end]`로 첫 행부터 `end` 직전 행까지 고른다. 끝 위치는 포함되지 않으므로 10개 행이 선택된다.

```python
end = int(len(df) * 0.7)
df.iloc[:end]     # 앞 70%
df.iloc[end:]     # 나머지 30%
```

공식 문서: [iloc](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.iloc.html) · [int() 내장 함수](https://docs.python.org/ko/3.13/library/functions.html#int)

In [ ]:
# [예제 9] iloc: 앞에서부터 70%만 고르기
# 1. 전체 행 개수에 0.7을 곱하여 70%에 해당하는 행 개수를 계산한다.
#    int()는 소수점 이하를 버리고 정수로 변환한다.
#    예: 15 × 0.7 = 10.5 → 10
end = int(len(df) * 0.7)

# 2. iloc[:end]로 첫 번째 행부터 end 직전 행까지 선택한다.
#    iloc는 끝 위치를 포함하지 않으므로 0~9번째 행, 총 10개 행을 선택한다.
print(end, df.iloc[:end].shape)

# 3. 나머지 행은 iloc[end:]로 고른다.
print(df.iloc[end:].shape)

### **[실습] 4. iloc**


In [ ]:
# [실습] iloc: 정수 위치로 조회
# 1. 첫 번째~세 번째 행, 첫 번째와 세 번째 열을 조회하시오.
# 2. 마지막 행에서 두 번째 열의 값 하나를 출력하시오.
# 힌트: 위치는 0부터 시작한다. 슬라이싱 끝 위치는 제외한다.


## **5) 조건 필터링**

조건에 맞는 행만 골라내는 방법이다. 대괄호 안에 조건식을 넣는다.

```python
df[조건식]
```

`df["매출"] >= 1000000`을 실행하면 값이 아니라 행 인덱스가 유지되는 불리언 Series(`True`/`False`)가 나온다. 이 목록을 대괄호에 넣으면 참인 행만 남는다.

```text
 df["매출"] >= 1000000   →  [True, False, True, ...]

 df[위 결과]              →  True인 행만 조회된다
```

NumPy에서 배운 Boolean Indexing과 원리가 같다. 배열 대신 표에 적용되는 것뿐이다.

공식 문서: [Boolean indexing](https://pandas.pydata.org/docs/user_guide/indexing.html#boolean-indexing)


In [ ]:
# [예제] 조건에 맞는 행만 고르기
df[df["매출"] >= 1000000]


,주문번호,주문일,지역,제품군,제품명,수량,매출
0,O001,2026-01-05,서울,전자,노트북,2,2400000
3,O004,2026-02-03,대전,전자,모니터,4,1200000
8,O009,2026-03-19,서울,전자,태블릿,3,2100000
10,O011,2026-04-04,부산,전자,이어폰,9,1350000
13,O014,2026-05-03,서울,전자,스마트폰,4,3200000
14,O015,2026-05-18,부산,가구,소파,1,1100000


### **조건 필터링: 비교 연산자별 조회**

`==`, `!=`, `>`, `>=`, `<`, `<=`를 비교한다. 조건이 일치하는 행이 없어도 빈 DataFrame은 정상 결과다.


In [ ]:
# [예제] 비교 연산자를 바꾸어 조회하기
# 1. 매출이 100만 원 초과인 행을 조회한다.
display(df[df["매출"] > 1000000])

print('\n')
# 2. 매출이 100만 원 이하인 행을 조회한다.
display(df[df["매출"] <= 1000000])
print('\n')
# 3. 지역이 서울이 아닌 행을 조회한다.
display(df[df["지역"] != "서울"])
print('\n')
# 4. 비교 결과(True/False)를 합산하여 조건에 맞는 행 수를 구한다.
print((df["매출"] >= 1000000).sum())

,주문번호,주문일,지역,제품군,제품명,수량,매출
0,O001,2026-01-05,서울,전자,노트북,2,2400000
3,O004,2026-02-03,대전,전자,모니터,4,1200000
8,O009,2026-03-19,서울,전자,태블릿,3,2100000
10,O011,2026-04-04,부산,전자,이어폰,9,1350000
13,O014,2026-05-03,서울,전자,스마트폰,4,3200000
14,O015,2026-05-18,부산,가구,소파,1,1100000


,주문번호,주문일,지역,제품군,제품명,수량,매출
1,O002,2026-01-12,부산,가구,책상,3,900000
2,O003,2026-01-20,서울,식품,커피,10,300000
4,O005,2026-02-14,부산,식품,간식세트,8,480000
5,O006,2026-02-25,서울,가구,의자,5,750000
6,O007,2026-03-02,대전,전자,키보드,7,560000
7,O008,2026-03-11,부산,가구,수납장,2,800000
9,O010,2026-03-28,대전,식품,차세트,6,420000
11,O012,2026-04-15,서울,가구,테이블,2,700000
12,O013,2026-04-22,대전,식품,건강식품,5,550000


,주문번호,주문일,지역,제품군,제품명,수량,매출
1,O002,2026-01-12,부산,가구,책상,3,900000
3,O004,2026-02-03,대전,전자,모니터,4,1200000
4,O005,2026-02-14,부산,식품,간식세트,8,480000
6,O007,2026-03-02,대전,전자,키보드,7,560000
7,O008,2026-03-11,부산,가구,수납장,2,800000
9,O010,2026-03-28,대전,식품,차세트,6,420000
10,O011,2026-04-04,부산,전자,이어폰,9,1350000
12,O013,2026-04-22,대전,식품,건강식품,5,550000
14,O015,2026-05-18,부산,가구,소파,1,1100000




6


### **[실습] 5. 조건 필터링**


In [ ]:
# [실습] 조건 필터링: 비교 연산자로 행 조회
# 1. 매출이 1000000 이상인 행을 조회하시오.
# 2. 그 조건을 만족하는 행의 개수를 출력하시오.
# 힌트: 조건식의 True는 합계 계산에서 1로 취급된다.


## **6) 다중 조건**

조건을 두 개 이상 함께 쓸 때는 다음 기호를 사용한다.

| 기호 | 뜻 | 파이썬의 대응 표현 |
|---|---|---|
| `&` | 그리고 | `and` |
| `\|` | 또는 | `or` |
| `~` | 반대 | `not` |

### **and와 or를 쓸 수 없는 이유**
조건을 연결할 때 `and`와 `or`를 사용했다. 그런데 Pandas에서는 이 방식이 통하지 않고 오류가 발생한다.`df["매출"] >= 1000000`의 결과는 값 하나가 아니라 행마다 참과 거짓이 담긴 목록이다. 목록 전체가 참인지 거짓인지는 정할 수 없으므로 파이썬이 판단하지 못하고 오류를 낸다.

```text
 df["매출"] >= 1000000   →  [True, False, True, ...]   행마다 하나씩

 and 는 이 목록 전체가 참인지 물어본다   →  판단 불가, 오류
 &  는 행마다 하나씩 비교한다            →  정상 동작
```

### **각 조건을 괄호로 묶는 이유**

`&`는 `>=`나 `==` 같은 비교 기호보다 먼저 계산된다. 괄호가 없으면 파이썬이 엉뚱한 순서로 계산해 오류가 발생하므로, 각 조건을 반드시 괄호로 묶는다.

```python
df[df["지역"] == "서울" & df["제품군"] == "전자"]     # 오류
df[(df["지역"] == "서울") & (df["제품군"] == "전자")]  # 정상
```

조건이 길어지면 변수에 담아 두면 읽기 쉬워진다.


In [ ]:
# [예제] 조건 여러 개 묶기 (&)
condition = (
    (df["지역"] == "서울")
    & (df["제품군"] == "전자")
    & (df["매출"] >= 1000000)
)

df[condition]


,주문번호,주문일,지역,제품군,제품명,수량,매출
0,O001,2026-01-05,서울,전자,노트북,2,2400000
8,O009,2026-03-19,서울,전자,태블릿,3,2100000
13,O014,2026-05-03,서울,전자,스마트폰,4,3200000


In [ ]:
# [ 예제] 다중 조건: 그리고(&), 또는(|)
mini = pd.DataFrame({"지역": ["서울", "부산", "서울", "인천"], "수량": [8, 3, 5, 9]})
display(mini[(mini["지역"] == "서울") & (mini["수량"] >= 5)])  # 서울이고 수량 5 이상
display(mini[(mini["지역"] == "서울") | (mini["수량"] >= 9)])  # 서울 또는 수량 9 이상

,지역,수량
0,서울,8
2,서울,5


,지역,수량
0,서울,8
2,서울,5
3,인천,9


### **다중 조건: AND·OR·NOT**

각 비교식을 괄호로 묶는다. `&`는 모두 만족, `|`는 하나 이상 만족, `~`는 조건 반대다.


### **[실습] 6. 다중 조건을 활용한 데이터 조회**

 1. 제품군이 '가구'이면서 매출이 500000 이상인 행을 조회하시오.


 2. 제품군이 '가구' 또는 '식품'인 행을 조회하시오.


 3. 제품군이 '가구'가 아니면서 매출이 300000 이상인 행을 조회하시오.


 4. 제품군이 '가구' 또는 '식품'이면서
    매출이 500000 이상인 행을 조회하시오.


 5. 매출이 300000 미만이거나 1000000 초과인 행을 조회하시오.

In [ ]:
# [실습] 6. 다중 조건을 활용한 데이터 조회

# 1. 제품군이 '가구'이면서 매출이 500000 이상인 행을 조회하시오.


# 2. 제품군이 '가구' 또는 '식품'인 행을 조회하시오.


# 3. 제품군이 '가구'가 아니면서 매출이 300000 이상인 행을 조회하시오.


# 4. 제품군이 '가구' 또는 '식품'이면서
#    매출이 500000 이상인 행을 조회하시오.


# 5. 매출이 300000 미만이거나 1000000 초과인 행을 조회하시오.


## **7) 문자열로 조건 쓰기 `query()`**

`query()`는 조건식을 문자열로 적어 행을 고른다. 열 이름을 `df["열"]` 대신 그대로 적고, `&`·`|` 대신 `and`·`or`를 쓸 수 있어 조건이 길 때 읽기 쉽다.

```python
# 조건 필터링
df[(df["지역"] == "서울") & (df["매출"] >= 1000000)]

# query(): 같은 결과
df.query('지역 == "서울" and 매출 >= 1000000')
```

| 규칙 | 조건 필터링 `df[...]` | `query()` |
|---|---|---|
| 열 이름 | `df["지역"]` | `지역` |
| 문자값 | `"서울"` | `"서울"` |
| 그리고 / 또는 / 반대 | `&` · `\|` · `~` + 괄호 | `and` · `or` · `not` |
| 숫자 비교 | `df["매출"] >= 1000000` | `매출 >= 1000000` |
| 변수 사용 | `df["매출"] >= minimum` | `매출 >= @minimum` |

- 조건 전체는 작은따옴표 `'...'`로 감싸고, 안의 글자값은 큰따옴표 `"..."`로 쓴다. 같은 따옴표를 겹쳐 쓰면 문자열이 중간에 끝나 `SyntaxError`가 난다.
- 파이썬 변수의 값을 쓰려면 변수 이름 앞에 `@`를 붙인다.
- 열 이름에 띄어쓰기가 있으면(`Order Date`) 백틱(`` ` ``)으로 감싼다. 예: ``orders.query('`Ship Mode` == "First Class"')``

공식 문서: [DataFrame.query()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.query.html)


In [ ]:
# [예제] query() 로 조회하기
display(df.query('지역 == "서울" and 매출 >= 1000000'))   # 서울이면서 매출 100만 이상
display(df.query('제품군 == "가구" or 수량 >= 9'))         # 가구이거나 수량 9개 이상

minimum = 2000000
display(df.query('매출 >= @minimum'))                     # 변수 값은 @ 를 붙여 쓴다

,주문번호,주문일,지역,제품군,제품명,수량,매출
0,O001,2026-01-05,서울,전자,노트북,2,2400000
8,O009,2026-03-19,서울,전자,태블릿,3,2100000
13,O014,2026-05-03,서울,전자,스마트폰,4,3200000


,주문번호,주문일,지역,제품군,제품명,수량,매출
1,O002,2026-01-12,부산,가구,책상,3,900000
2,O003,2026-01-20,서울,식품,커피,10,300000
5,O006,2026-02-25,서울,가구,의자,5,750000
7,O008,2026-03-11,부산,가구,수납장,2,800000
10,O011,2026-04-04,부산,전자,이어폰,9,1350000
11,O012,2026-04-15,서울,가구,테이블,2,700000
14,O015,2026-05-18,부산,가구,소파,1,1100000


,주문번호,주문일,지역,제품군,제품명,수량,매출
0,O001,2026-01-05,서울,전자,노트북,2,2400000
8,O009,2026-03-19,서울,전자,태블릿,3,2100000
13,O014,2026-05-03,서울,전자,스마트폰,4,3200000


### **query(): 조건별 문자열 조회**

`query()`의 조건은 문자열로 작성한다. 외부 Python 변수는 `@변수명`으로 참조한다.


In [ ]:
# [예제] query()로 조건별 조회하기
# 1. 특정 지역 조회
display(df.query('지역 == "서울"'))

# 2. 매출 범위 조회
display(df.query("매출 >= 500000 and 매출 <= 1500000"))

# 3. OR 조건 조회
display(df.query('지역 == "서울" or 지역 == "부산"'))

# 4. 외부 변수 조회: @min_sales는 문자열 안에서 Python 변수 값을 참조한다.
min_sales = 1000000
display(df.query("매출 >= @min_sales"))

,주문번호,주문일,지역,제품군,제품명,수량,매출
0,O001,2026-01-05,서울,전자,노트북,2,2400000
2,O003,2026-01-20,서울,식품,커피,10,300000
5,O006,2026-02-25,서울,가구,의자,5,750000
8,O009,2026-03-19,서울,전자,태블릿,3,2100000
11,O012,2026-04-15,서울,가구,테이블,2,700000
13,O014,2026-05-03,서울,전자,스마트폰,4,3200000


,주문번호,주문일,지역,제품군,제품명,수량,매출
1,O002,2026-01-12,부산,가구,책상,3,900000
3,O004,2026-02-03,대전,전자,모니터,4,1200000
5,O006,2026-02-25,서울,가구,의자,5,750000
6,O007,2026-03-02,대전,전자,키보드,7,560000
7,O008,2026-03-11,부산,가구,수납장,2,800000
10,O011,2026-04-04,부산,전자,이어폰,9,1350000
11,O012,2026-04-15,서울,가구,테이블,2,700000
12,O013,2026-04-22,대전,식품,건강식품,5,550000
14,O015,2026-05-18,부산,가구,소파,1,1100000


,주문번호,주문일,지역,제품군,제품명,수량,매출
0,O001,2026-01-05,서울,전자,노트북,2,2400000
1,O002,2026-01-12,부산,가구,책상,3,900000
2,O003,2026-01-20,서울,식품,커피,10,300000
4,O005,2026-02-14,부산,식품,간식세트,8,480000
5,O006,2026-02-25,서울,가구,의자,5,750000
7,O008,2026-03-11,부산,가구,수납장,2,800000
8,O009,2026-03-19,서울,전자,태블릿,3,2100000
10,O011,2026-04-04,부산,전자,이어폰,9,1350000
11,O012,2026-04-15,서울,가구,테이블,2,700000
13,O014,2026-05-03,서울,전자,스마트폰,4,3200000


,주문번호,주문일,지역,제품군,제품명,수량,매출
0,O001,2026-01-05,서울,전자,노트북,2,2400000
3,O004,2026-02-03,대전,전자,모니터,4,1200000
8,O009,2026-03-19,서울,전자,태블릿,3,2100000
10,O011,2026-04-04,부산,전자,이어폰,9,1350000
13,O014,2026-05-03,서울,전자,스마트폰,4,3200000
14,O015,2026-05-18,부산,가구,소파,1,1100000


### **[실습] 7. query()**


In [ ]:
# [실습] query(): 문자열로 조건 조회
# 1. 지역이 서울인 행을 query()로 조회하시오.
# 2. minimum = 1000000 변수를 만든 뒤 매출이 그 이상인 행을 조회하시오.
# 힌트: query() 안에서 바깥 Python 변수는 @변수명으로 쓴다.


## **8) 조건으로 값 바꾸기 `loc[조건, 열] = 값`**

조건에 맞는 행만 골라 **특정 열의 값을 바꿀** 때는 `loc`의 행 자리에 조건식, 열 자리에 열 이름을 쓰고 `=`로 값을 넣는다.

```python
df.loc[df["매출"] >= 1000000, "등급"] = "고액"
```

| 작성 | 코드 | 결과 |
|---|---|---|
| 새 열에 기본값 넣기 | `df["등급"] = "일반"` | 모든 행의 등급이 "일반" |
| 조건이 참인 행만 바꾸기 | `df.loc[df["매출"] >= 1000000, "등급"] = "고액"` | 매출 100만 이상인 행만 "고액" |
| 여러 열을 한 번에 | `df.loc[df["지역"] == "대전", ["수량", "매출"]] = 0` | 대전 행의 수량·매출이 0 |


공식 문서: [DataFrame.loc](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.loc.html) · [인덱싱과 선택 안내](https://pandas.pydata.org/docs/user_guide/indexing.html)


In [ ]:
# [예제] 조건에 맞는 행만 값 바꾸기
graded = df[["주문번호", "지역", "수량", "매출"]].copy()    # 원본 df 를 바꾸지 않도록 복사본으로 연습한다

graded["등급"] = "일반"                                    # 1. 모든 행에 기본값
graded.loc[graded["매출"] >= 1000000, "등급"] = "고액"     # 2. 매출 100만 이상인 행만 "고액"
display(graded.head())
print(graded["등급"].value_counts())                       # 고액·일반 건수

graded.loc[graded["지역"] == "대전", ["수량", "매출"]] = 0  # 여러 열을 한 번에: 대전 행의 수량·매출을 0으로
display(graded[graded["지역"] == "대전"])

,주문번호,지역,수량,매출,등급
0,O001,서울,2,2400000,고액
1,O002,부산,3,900000,일반
2,O003,서울,10,300000,일반
3,O004,대전,4,1200000,고액
4,O005,부산,8,480000,일반


등급
일반    9
고액    6
Name: count, dtype: int64


,주문번호,지역,수량,매출,등급
3,O004,대전,0,0,고액
6,O007,대전,0,0,일반
9,O010,대전,0,0,일반
12,O013,대전,0,0,일반


### **loc 값 변경**



In [ ]:
# [예제] 전체 열·특정 셀·조건에 맞는 셀 수정
edit_df = df[["지역", "제품군", "매출"]].copy()  # 원본 보호

# 1. 연습용 열 전체에 같은 값을 넣는다.
edit_df.loc[:, "확인상태"] = "미확인"

display(edit_df.head(5))
# 2. 첫 번째 행의 특정 열만 변경한다.
#    loc[행 레이블, 열 이름]으로 한 개의 셀을 지정한다.
edit_df.loc[edit_df.index[0], "확인상태"] = "확인"

# 3. 매출 100만 원 이상인 행에만 상태를 입력한다.
edit_df.loc[edit_df["매출"] >= 1000000, "확인상태"] = "고액 확인"

# 4. 변경 전 원본과 변경 후 복사본을 확인한다.
display(df.head(3))
display(edit_df.head(3))

,지역,제품군,매출,확인상태
0,서울,전자,2400000,미확인
1,부산,가구,900000,미확인
2,서울,식품,300000,미확인
3,대전,전자,1200000,미확인
4,부산,식품,480000,미확인


,주문번호,주문일,지역,제품군,제품명,수량,매출
0,O001,2026-01-05,서울,전자,노트북,2,2400000
1,O002,2026-01-12,부산,가구,책상,3,900000
2,O003,2026-01-20,서울,식품,커피,10,300000


,지역,제품군,매출,확인상태
0,서울,전자,2400000,고액 확인
1,부산,가구,900000,미확인
2,서울,식품,300000,미확인


### **[실습] 8. loc 값 변경**


In [ ]:
# [실습] loc 값 변경: 원본 보존하기
# 1. df의 '지역', '매출' 열을 복사하여 status_df에 저장하시오.
# 2. '구분' 열 전체에 '일반'을 넣으시오.
# 3. 매출이 1000000 이상인 행만 '고액'으로 바꾸시오.
# 4. '구분'의 값별 개수를 출력하시오.
# 힌트: .copy()로 원본을 보호하고 loc[조건, 열]에 대입한다.


## **9) isin()**

값이 여러 개 중 하나에 해당하는지 확인한다.

```python
df[df["열이름"].isin([값1, 값2])]
```

`or`로 조건을 여러 번 연결하는 것과 결과가 같지만 훨씬 간결하다. 확인할 값이 많아질수록 차이가 커진다.

```text
 (df["지역"] == "서울") | (df["지역"] == "부산")
 df["지역"].isin(["서울", "부산"])                  위와 같은 결과
```

앞에 `~`를 붙이면 목록에 없는 데이터를 조회한다.

공식 문서: [isin()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.isin.html)


In [ ]:
# [예제] isin(): 여러 값 중 하나
df[df["지역"].isin(["서울", "부산"])]


,주문번호,주문일,지역,제품군,제품명,수량,매출
0,O001,2026-01-05,서울,전자,노트북,2,2400000
1,O002,2026-01-12,부산,가구,책상,3,900000
2,O003,2026-01-20,서울,식품,커피,10,300000
4,O005,2026-02-14,부산,식품,간식세트,8,480000
5,O006,2026-02-25,서울,가구,의자,5,750000
7,O008,2026-03-11,부산,가구,수납장,2,800000
8,O009,2026-03-19,서울,전자,태블릿,3,2100000
10,O011,2026-04-04,부산,전자,이어폰,9,1350000
11,O012,2026-04-15,서울,가구,테이블,2,700000
13,O014,2026-05-03,서울,전자,스마트폰,4,3200000


In [ ]:
# [예제] isin(): 여러 값 중 하나 찾기
mini = pd.DataFrame({"제품": ["A", "B", "C", "D"], "지역": ["서울", "부산", "인천", "서울"]})
display(mini[mini["지역"].isin(["서울", "부산"])])

,제품,지역
0,A,서울
1,B,부산
3,D,서울


### **isin(): 포함·제외·열 선택**

여러 후보 값 중 하나와 일치하는 행을 조회한다. `~`를 붙이면 후보에 없는 값이 선택된다.


In [ ]:
# [예제] isin() 조건을 다양한 방식으로 사용하기
# 1. 서울 또는 부산인 행
display(df[df["지역"].isin(["서울", "부산"])])

# 2. 서울·부산 이외의 행
display(df[~df["지역"].isin(["서울", "부산"])])

# 3. 조건에 맞는 행의 지역과 매출 열만 조회한다.
display(df.loc[df["지역"].isin(["서울", "부산"]), ["지역", "매출"]])

,주문번호,주문일,지역,제품군,제품명,수량,매출
0,O001,2026-01-05,서울,전자,노트북,2,2400000
1,O002,2026-01-12,부산,가구,책상,3,900000
2,O003,2026-01-20,서울,식품,커피,10,300000
4,O005,2026-02-14,부산,식품,간식세트,8,480000
5,O006,2026-02-25,서울,가구,의자,5,750000
7,O008,2026-03-11,부산,가구,수납장,2,800000
8,O009,2026-03-19,서울,전자,태블릿,3,2100000
10,O011,2026-04-04,부산,전자,이어폰,9,1350000
11,O012,2026-04-15,서울,가구,테이블,2,700000
13,O014,2026-05-03,서울,전자,스마트폰,4,3200000


,주문번호,주문일,지역,제품군,제품명,수량,매출
3,O004,2026-02-03,대전,전자,모니터,4,1200000
6,O007,2026-03-02,대전,전자,키보드,7,560000
9,O010,2026-03-28,대전,식품,차세트,6,420000
12,O013,2026-04-22,대전,식품,건강식품,5,550000


,지역,매출
0,서울,2400000
1,부산,900000
2,서울,300000
4,부산,480000
5,서울,750000
7,부산,800000
8,서울,2100000
10,부산,1350000
11,서울,700000
13,서울,3200000


### **[실습] 9. isin()**


In [ ]:
# [실습] isin(): 목록에 포함되는 값 찾기
# 1.지역이 '서울', '대전' 중 하나인 행을 조회하시오.
# 2.지역이 '서울', '대전'이 아닌 행의 '지역', '매출' 열만 조회하시오.
# 3.지역이 '서울', '대전' 중 하나이면서 매출이 1000000 이상인 행을 조회하시오.


## **10) between()**

지정한 두 값 사이의 데이터를 조회한다.

```python
df[df["열이름"].between(최솟값, 최댓값)]
```

기본적으로 **양쪽 끝 값이 모두 포함된다.** `between(500000, 1500000)`은 50만 원과 150만 원도 결과에 들어간다.

포함 여부를 바꾸려면 `inclusive`를 지정한다.

| 지정값 | 범위 |
|---|---|
| `"both"` (기본값) | 최솟값 이상, 최댓값 이하 |
| `"left"` | 최솟값 이상, 최댓값 미만 |
| `"right"` | 최솟값 초과, 최댓값 이하 |
| `"neither"` | 최솟값 초과, 최댓값 미만 |

날짜형으로 변환한 열에도 사용할 수 있다.

공식 문서: [between()](https://pandas.pydata.org/docs/reference/api/pandas.Series.between.html)


In [ ]:
# [예제] between(): 범위 안의 값
df[df["매출"].between(500000, 1500000)]


,주문번호,주문일,지역,제품군,제품명,수량,매출
1,O002,2026-01-12,부산,가구,책상,3,900000
3,O004,2026-02-03,대전,전자,모니터,4,1200000
5,O006,2026-02-25,서울,가구,의자,5,750000
6,O007,2026-03-02,대전,전자,키보드,7,560000
7,O008,2026-03-11,부산,가구,수납장,2,800000
10,O011,2026-04-04,부산,전자,이어폰,9,1350000
11,O012,2026-04-15,서울,가구,테이블,2,700000
12,O013,2026-04-22,대전,식품,건강식품,5,550000
14,O015,2026-05-18,부산,가구,소파,1,1100000


### **between(): 경계값 포함 여부**

`inclusive`는 양쪽 경계값을 포함하는지 정한다. `both`, `left`, `right`, `neither`를 비교한다.


In [ ]:
# [예제] between()의 경계 포함 옵션 비교
# 값 100과 200이 실제로 존재하는 예시를 만든다.
range_ex = pd.DataFrame({"거래": ["A", "B", "C", "D"], "금액": [90, 100, 150, 200]})

# 1. 양쪽 포함: 100 이상 200 이하
display(range_ex[range_ex["금액"].between(100, 200, inclusive="both")])

# 2. 왼쪽만 포함: 100 이상 200 미만
display(range_ex[range_ex["금액"].between(100, 200, inclusive="left")])

# 3. 오른쪽만 포함: 100 초과 200 이하
display(range_ex[range_ex["금액"].between(100, 200, inclusive="right")])

# 4. 양쪽 제외: 100 초과 200 미만
display(range_ex[range_ex["금액"].between(100, 200, inclusive="neither")])

,거래,금액
1,B,100
2,C,150
3,D,200


,거래,금액
1,B,100
2,C,150


,거래,금액
2,C,150
3,D,200


,거래,금액
2,C,150


날짜에도 `between()`을 사용할 수 있다.


In [ ]:
# [예제] 날짜 범위로 고르기
df[df["주문일"].between("2026-02-01", "2026-03-31")]


,주문번호,주문일,지역,제품군,제품명,수량,매출
3,O004,2026-02-03,대전,전자,모니터,4,1200000
4,O005,2026-02-14,부산,식품,간식세트,8,480000
5,O006,2026-02-25,서울,가구,의자,5,750000
6,O007,2026-03-02,대전,전자,키보드,7,560000
7,O008,2026-03-11,부산,가구,수납장,2,800000
8,O009,2026-03-19,서울,전자,태블릿,3,2100000
9,O010,2026-03-28,대전,식품,차세트,6,420000


In [ ]:
# [예제] 날짜·시간 범위 조회: 종료일 다음 날 0시 미만 사용
# 날짜형 열에 시간까지 들어 있으면 종료일만 지정한 between()은 마지막 날의 일부 주문을 놓칠 수 있다.
import pandas as pd

test_df = pd.DataFrame({
    "주문일": pd.to_datetime([
        "2026-02-01 09:00:00",
        "2026-03-15 12:30:00",
        "2026-03-31 00:00:00",
        "2026-03-31 14:30:00",
        "2026-03-31 23:59:59.500",
        "2026-04-01 09:00:00",
    ], format="ISO8601"),
    "매출": [100, 200, 300, 400, 500, 600]
})

# 1. 종료일 자정까지만 포함: 결과 3건
result1 = test_df[test_df["주문일"].between("2026-02-01", "2026-03-31")]
print("종료일만 지정:", len(result1))  # 3

display(result1)

# 2. 다음 날 0시 직전까지: 3월 31일 모든 시각 포함 (5건)
result2 = test_df[
    (test_df["주문일"] >= "2026-02-01") &
    (test_df["주문일"] < "2026-04-01")
]
print("다음 날 0시 미만:", len(result2))  # 5
display(result2)
result3 = test_df[test_df["주문일"].between("2026-02-01", "2026-04-01")]
print("다음 날 0시 미만2:", len(result3))  # 3
display(result3)
# 같은 날 23:59:59.500처럼 소수점 이하 초가 있어도 누락되지 않는다.


종료일만 지정: 3


,주문일,매출
0,2026-02-01 09:00:00,100
1,2026-03-15 12:30:00,200
2,2026-03-31 00:00:00,300


다음 날 0시 미만: 5


,주문일,매출
0,2026-02-01 09:00:00.000,100
1,2026-03-15 12:30:00.000,200
2,2026-03-31 00:00:00.000,300
3,2026-03-31 14:30:00.000,400
4,2026-03-31 23:59:59.500,500


### **[실습] 10. between()**


In [ ]:
# [실습] between(): 점수 범위 조회하기
# 다음은 학생별 시험 점수이다.
score_df = pd.DataFrame({
    "학생": ["A", "B", "C", "D", "E"],
    "점수": [55, 60, 75, 90, 95]
})

# 1. 점수가 60점 이상 90점 이하인 학생을 조회하시오.
# 2. 점수가 60점 초과 90점 이하인 학생을 조회하시오.
# 3. 점수가 60~90점 범위에 포함되지 않는 학생을 조회하시오.

## **11) 문자열 포함 조회 `str.contains()`**

`==`는 값이 완전히 같을 때만 참이다. 제품명에 특정 글자가 들어 있는 행을 찾을 때는 `.str.contains()`를 쓴다.

```python
df[df["열이름"].str.contains("찾을글자")]
```

| 옵션 | 뜻 |
|---|---|
| `case=False` | 영문 대소문자를 구분하지 않는다 |
| `na=False` | 결측값을 `False`(거짓)로 처리한다 |


```python
# 결측값을 False로 처리하여 안전하게 조건 조회
df[df["제품명"].str.contains("모터", na=False)]
```

`.str.startswith("글자")`는 그 글자로 시작하는 값, `.str.endswith("글자")`는 그 글자로 끝나는 값을 찾는다.

공식 문서: [Series.str.contains()](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.contains.html)


In [ ]:
# [예제] 제품명에 "세트"가 들어 있는 주문
display(df[df["제품명"].str.contains("세트")])
# 영문은 대소문자를 구분하지 않도록 case=False
display(orders[orders["Product Name"].str.contains("chair", case=False)].head(3))


,주문번호,주문일,지역,제품군,제품명,수량,매출
4,O005,2026-02-14,부산,식품,간식세트,8,480000
9,O010,2026-03-28,대전,식품,차세트,6,420000


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country/Region,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
5,6,US-2023-167199,2023-01-06,2023-01-10,Standard Class,ME-17320,Maria Etezadi,Home Office,United States,Henderson,...,42420,South,FUR-CH-10004063,Furniture,Chairs,Global Deluxe High-Back Manager's Chair,2573.82,9,0.0,746.4078
43,44,US-2023-147627,2023-01-20,2023-01-26,Standard Class,HL-15040,Hunter Lopez,Consumer,United States,Jonesboro,...,72401,South,FUR-CH-10002331,Furniture,Chairs,Hon 4700 Series Mobuis Mid-Back Task Chairs wi...,1067.94,3,0.0,224.2674
45,46,US-2023-167927,2023-01-20,2023-01-26,Standard Class,XP-21865,Xylona Preis,Consumer,United States,Westland,...,48185,Central,FUR-FU-10002918,Furniture,Furnishings,Eldon ClusterMat Chair Mat with Cordless Antis...,272.94,3,0.0,30.0234


### **문자열 조회: contains·startswith·endswith**

문자열이 포함되는지, 시작하는지, 끝나는지를 구분한다. `na=False`는 결측값을 조건 불일치로 처리한다.


In [ ]:
# [예제] 제품명 문자열의 위치에 따라 조회하기
# 1. 제품명에 '세트'가 포함된 행을 조회한다.
print('1')
display(df[df["제품명"].str.contains("세트", na=False)])
print('\n')
# 2. 제품명이 '스'로 시작하는 행을 조회한다.
print('2')
display(df[df["제품명"].str.startswith("스", na=False)])
print('\n')
# 3. 제품명이 '세트'로 끝나는 행을 조회한다.
print('3')
display(df[df["제품명"].str.endswith("세트", na=False)])
print('\n')
# 4. 제품명에 '세트'가 없는 행을 조회한다.
print('4')
display(df[~df["제품명"].str.contains("세트", na=False)])
print('\n')
# 5. 대소문자 무시: 영문 상품명 예시
print('5')
display(orders[orders["Product Name"].str.contains("table", case=False, na=False)].head())

1


,주문번호,주문일,지역,제품군,제품명,수량,매출
4,O005,2026-02-14,부산,식품,간식세트,8,480000
9,O010,2026-03-28,대전,식품,차세트,6,420000




2


,주문번호,주문일,지역,제품군,제품명,수량,매출
13,O014,2026-05-03,서울,전자,스마트폰,4,3200000




3


,주문번호,주문일,지역,제품군,제품명,수량,매출
4,O005,2026-02-14,부산,식품,간식세트,8,480000
9,O010,2026-03-28,대전,식품,차세트,6,420000




4


,주문번호,주문일,지역,제품군,제품명,수량,매출
0,O001,2026-01-05,서울,전자,노트북,2,2400000
1,O002,2026-01-12,부산,가구,책상,3,900000
2,O003,2026-01-20,서울,식품,커피,10,300000
3,O004,2026-02-03,대전,전자,모니터,4,1200000
5,O006,2026-02-25,서울,가구,의자,5,750000
6,O007,2026-03-02,대전,전자,키보드,7,560000
7,O008,2026-03-11,부산,가구,수납장,2,800000
8,O009,2026-03-19,서울,전자,태블릿,3,2100000
10,O011,2026-04-04,부산,전자,이어폰,9,1350000
11,O012,2026-04-15,서울,가구,테이블,2,700000




5


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country/Region,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
43,44,US-2023-147627,2023-01-20,2023-01-26,Standard Class,HL-15040,Hunter Lopez,Consumer,United States,Jonesboro,...,72401,South,FUR-CH-10002331,Furniture,Chairs,Hon 4700 Series Mobuis Mid-Back Task Chairs wi...,1067.94,3,0.0,224.2674
65,66,CA-2023-115238,2023-01-21,2023-01-24,Standard Class,JW-15220,Jane Waco,Corporate,Canada,Toronto,...,M7A,East,TEC-PH-10001300,Technology,Phones,iKross Bluetooth Portable Keyboard + Cell Phon...,209.50,10,0.0,58.6600
77,78,US-2023-117163,2023-01-27,2023-02-02,Standard Class,EJ-13720,Ed Jacobs,Consumer,United States,San Diego,...,92037,West,FUR-TA-10003469,Furniture,Tables,Balt Split Level Computer Training Table,333.00,3,0.2,-16.6500
89,90,US-2023-111059,2023-02-03,2023-02-06,Second Class,TB-21400,Tom Boeckenhauer,Consumer,United States,Seattle,...,98105,West,OFF-BI-10004593,Office Supplies,Binders,Ibico Laser Imprintable Binding System Covers,83.84,2,0.2,27.2480
102,103,US-2023-127614,2023-02-11,2023-02-15,Standard Class,NF-18385,Natalie Fritzler,Consumer,United States,Chesapeake,...,23320,South,FUR-TA-10003715,Furniture,Tables,Hon 2111 Invitation Series Corner Table,1256.22,6,0.0,75.3732


### **[실습] 11. 문자열 검색**


In [ ]:
# [실습] 문자열 검색: 포함과 시작
# 1. df에서 제품명에 '세트'가 포함된 행을 조회하시오.
# 2. 제품명이 '스'로 시작하는 행을 조회하시오.
# 힌트: 결측값이 있어도 조회하도록 na=False를 사용한다.


## **12) 상위·하위 n개 `nlargest()` / `nsmallest()`**

"매출 상위 10건"처럼 큰 값 몇 개만 볼 때는 정렬 후 `head()`를 쓰는 대신 `nlargest()`를 쓰면 한 번에 끝난다.

```python
df.nlargest(3, "매출")     # 매출이 큰 순서로 3개 행
df.nsmallest(3, "매출")    # 매출이 작은 순서로 3개 행
```

`df.sort_values("매출", ascending=False).head(3)`과 비슷한 목적에 사용하지만 **동점 행의 선택·순서**는 옵션에 따라 다를 수 있다. `keep="all"`을 지정하면 동점을 모두 포함하여 n개보다 많아질 수 있다.

공식 문서: [nlargest()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.nlargest.html) · [nsmallest()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.nsmallest.html)


In [ ]:
# [예제] 매출 상위 3건과 하위 2건
display(df.nlargest(3, "매출"))
display(df.nsmallest(2, "매출")[["제품명", "매출"]])


,주문번호,주문일,지역,제품군,제품명,수량,매출
13,O014,2026-05-03,서울,전자,스마트폰,4,3200000
0,O001,2026-01-05,서울,전자,노트북,2,2400000
8,O009,2026-03-19,서울,전자,태블릿,3,2100000


,제품명,매출
2,커피,300000
9,차세트,420000


### **nlargest·nsmallest: 상하위 조회**

정렬 기준 열과 반환 개수를 달리해 비교한다. 같은 값이 존재하면 동점 처리 방식도 고려해야 한다.


In [ ]:
# [예제] 상위와 하위 데이터를 다양한 열로 조회하기
# 1. 매출 상위 5건 중 필요한 열만 선택한다.
display(df.nlargest(5, "매출")[["지역", "제품명", "매출"]])

# 2. 매출 하위 3건을 조회한다.
display(df.nsmallest(3, "매출"))

# 3. 주문 데이터에서 이익(Profit) 하위 3건을 조회한다.
display(orders.nsmallest(3, "Profit")[["Product Name", "Profit"]])

# 4. 동점 처리: 모든 동점 행을 포함하면 n개보다 많을 수 있다.
rank_ex = pd.DataFrame({"제품": ["A", "B", "C", "D"], "점수": [90, 90, 80, 70]})
display(rank_ex.nlargest(1, "점수", keep="all"))

,지역,제품명,매출
13,서울,스마트폰,3200000
0,서울,노트북,2400000
8,서울,태블릿,2100000
10,부산,이어폰,1350000
3,대전,모니터,1200000


,주문번호,주문일,지역,제품군,제품명,수량,매출
2,O003,2026-01-20,서울,식품,커피,10,300000
9,O010,2026-03-28,대전,식품,차세트,6,420000
4,O005,2026-02-14,부산,식품,간식세트,8,480000


,Product Name,Profit
6398,Cubify CubeX 3D Printer Double Head Print,-6599.9780
9318,Cubify CubeX 3D Printer Triple Head Print,-3839.9904
821,GBC DocuBind P400 Electric Binding System,-3701.8928


,제품,점수
0,A,90
1,B,90


## **13) 행 방향과 열 방향: `axis`**

`axis`는 **연산이 적용되는 축**을 나타낸다. 집계에서는 `axis=0`이 행 방향으로 내려가 각 열의 결과를 만들고, `axis=1`은 열 방향으로 이동하여 각 행의 결과를 만든다. `drop()`에서는 각각 행과 열을 제거한다. NumPy와 같은 규칙이다.

| 값 | 대상 | 결과 |
|---|---|---|
| `axis=0` | 행 방향(열축) | `sum()`에서는 열마다 결과를 계산하고, `drop()`에서는 행 삭제 |
| `axis=1` | 열 방향(행축) | `sum()`에서는 행마다 결과를 계산하고, `drop()`에서는 열 삭제 |

### **합계로 보기**

제품 3개의 1~3월 매출 표에서 `sum()`을 하면 다음과 같다.

```text
          1월   2월   3월          sum(axis=0)              sum(axis=1)
 A제품    100   150   200          행 3개가 하나로 합쳐짐       열 3개가 하나로 합쳐짐
 B제품    200   250   300          → 열마다 결과(월별 합계)     → 행마다 결과(제품별 1분기 합계)
 C제품    300   350   400
                                   1월   600                 A제품    450
                                   2월   750                 B제품    750
                                   3월   900                 C제품   1050
```

`axis=0`은 세로로 내려가며 더해 **열마다** 합계가 나온다. 결과의 이름표가 열 이름(1월·2월·3월)이다. `axis=1`은 가로로 옆으로 더해 **행마다** 합계가 나온다. 결과의 이름표가 행 이름(A제품·B제품·C제품)이다.



### **삭제로 보기**

`drop()`도 같은 규칙이다. `axis=0`이면 행이 사라지고, `axis=1`이면 열이 사라진다. 생략하면 `axis=0`(행 삭제)이다.

```python
df.drop(0)                    # axis=0 기본값: 0번 행이 사라진다
df.drop("매출", axis=1)        # 매출 열이 결과에서 제거된다
df.drop(columns=["매출"])     # 같은 뜻. 열 이름을 직접 지정하여 삭제
```





공식 문서: [DataFrame.sum()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sum.html) · [drop()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop.html)


In [ ]:
# [예제] axis=0 과 axis=1 비교
monthly = pd.DataFrame({
    "1월": [100, 200, 300],
    "2월": [150, 250, 350],
    "3월": [200, 300, 400]
}, index=["A제품", "B제품", "C제품"])
display(monthly)

# 1. 월별 매출 합계: 각 열의 합계 (세로 방향)
print(monthly.sum(axis=0))
# 1월    600
# 2월    750
# 3월    900

# 2. 제품별 1분기 매출 합계: 각 행의 합계 (가로 방향)
print(monthly.sum(axis=1))
# A제품     450
# B제품     750
# C제품    1050

,1월,2월,3월
A제품,100,150,200
B제품,200,250,300
C제품,300,350,400


1월    600
2월    750
3월    900
dtype: int64
A제품     450
B제품     750
C제품    1050
dtype: int64


### **axis=0·axis=1: 합계·평균·최댓값·삭제**

`axis=0`은 각 열을 따라 계산하여 열별 결과를 만들고, `axis=1`은 각 행을 따라 계산하여 행별 결과를 만든다.


In [ ]:
# [예제] axis에 따른 결과 비교
axis_ex = pd.DataFrame({"1분기": [100, 200, 300], "2분기": [150, 250, 350]}, index=["M01", "M02", "M03"])

# 1. 열별 합계: 각 열에서 행 방향으로 아래로 더한다.
print(axis_ex.sum(axis=0))

# 2. 행별 합계: 각 행에서 열 방향으로 옆으로 더한다.
print(axis_ex.sum(axis=1))

# 3. 열별 평균과 행별 최댓값을 비교한다.
print(axis_ex.mean(axis=0))
print(axis_ex.max(axis=1))

# 4. axis=0은 행 삭제, axis=1은 열 삭제에 사용한다.
display(axis_ex.drop("M01", axis=0))
display(axis_ex.drop("2분기", axis=1))

,1월,2월,3월
M01,4,5,3
M02,6,2,4
M03,3,7,5


1월    13
2월    14
3월    12
dtype: int64
M01    4.0
M02    4.0
M03    5.0
dtype: float64
M01    3
M02    2
M03    3
dtype: int64


,1월,2월,3월
M01,4,5,3
M03,3,7,5


,2월,3월
M01,5,3
M02,2,4
M03,7,5


### **[실습] 12. axis**


In [ ]:

# [실습] 12. axis를 활용한 데이터 계산
# 문제: 설비별 불량 건수 데이터를 이용하여 다음 작업을 수행하시오.

# 1. M01, M02, M03 설비의 월별 불량 건수 DataFrame을 생성하시오.
#    1월: [4, 6, 3], 2월: [5, 2, 7], 3월: [3, 4, 5]


# 2. 각 월의 전체 불량 건수를 구하시오. (axis=0)


# 3. 각 설비의 월평균 불량 건수를 구하시오. (axis=1)


# 4. 각 설비의 월별 불량 건수 중 최솟값을 구하시오. (axis=1)


# 5. M02 설비의 데이터를 제외하고 조회하시오. (axis=0)


# 6. 1월 데이터를 제외하고 조회하시오. (axis=1)



## **2. 자주 발생하는 오류**

## **KeyError**

```python
df["매촐"]
```

존재하지 않는 열 이름을 사용했을 때 발생한다.  
`df.columns`로 실제 열 이름을 확인한다.

## **FileNotFoundError**

```python
pd.read_csv("/content/drive/MyDrive/HD-AI/data/sale.csv")
```

파일명 또는 경로가 잘못됐을 때 발생한다. 위 예에서는 `sales.csv`를 `sale.csv`로 잘못 적었다. Google Drive가 연결되어 있지 않거나, 파일명만 적어서 `/content`에서 파일을 찾는 경우에도 발생한다. `os.path.exists()`로 경로를 먼저 확인한다.

## **UnicodeDecodeError**

CSV 파일의 인코딩과 `encoding` 옵션이 맞지 않을 때 발생한다.

## **TypeError**

날짜가 문자열인 상태에서 날짜 계산을 하거나, 숫자 열에 문자열이 섞여 있을 때 발생할 수 있다.

## **AttributeError**

```python
df.Sum()             # AttributeError: 'DataFrame' object has no attribute 'Sum'
df["주문일"].dt.year  # 주문일이 문자열이면 AttributeError: Can only use .dt accessor with datetimelike values
```

없는 메서드·속성 이름을 썼을 때 발생한다. 대소문자 오타(`Sum`, `Head`)가 가장 흔하다. `.dt`를 썼는데 이 오류가 나면 그 열이 아직 날짜형이 아니라는 뜻이므로 `pd.to_datetime()`을 먼저 한다. `.str`도 마찬가지로 문자열 열에만 쓸 수 있다.

## **괄호를 빼먹은 메서드 (오류 없음)**

```python
df["매출"].mean      # <bound method ...> 가 표시된다
df["매출"].mean()    # 평균이 계산된다
```

괄호가 없으면 오류는 나지 않지만 계산을 실행하지 않고 메서드 자체를 보여 준다. 숫자 대신 `<bound method ...>`가 보이면 괄호를 빠뜨린 것이다.

## **ValueError (자료형 변환)**

```python
df["가격"].astype(int)   # ValueError: invalid literal for int() with base 10: '1,200'
```

숫자로 바꿀 수 없는 글자(쉼표, 단위, 공백)가 섞여 있을 때 발생한다. `.str.replace()`로 먼저 없앤다.

## **조건식 오류**

```python
df[df["지역"] == "서울" & df["매출"] >= 1000000]
```

다중 조건에서는 각 조건을 괄호로 묶어야 한다.

```python
df[(df["지역"] == "서울") & (df["매출"] >= 1000000)]
```

## **SettingWithCopyWarning**

```python
df[df["매출"] >= 1000000]["등급"] = "고액"
```

연속 대입은 원본이 의도대로 변경되지 않을 수 있으며, pandas 버전에 따라 경고나 오류의 형태가 달라진다. `df.loc[조건, "열"] = 값`처럼 `loc` 한 번에 쓴다.

## **query()의 따옴표 오류 (SyntaxError)**

```python
df.query("지역 == "서울"")   # SyntaxError
```

바깥과 안쪽에 같은 따옴표를 쓰면 문자열이 중간에 끝난다. 바깥은 작은따옴표, 안쪽은 큰따옴표로 쓴다: `df.query('지역 == "서울"')`.

## **결측값이 있는 열의 정수 변환 (IntCastingNaNError)**

```python
df["수량"].astype(int)   # IntCastingNaNError: Cannot convert non-finite values (NA or inf) to integer
```

`NaN`이 남아 있는 열은 정수로 바꿀 수 없다. `fillna()`로 먼저 채운 뒤 `astype(int)`를 한다.


## **3. 확인문제**

pandas 2에서 배운 선택·조회 기능을 중심으로 확인한다.

1. `df["매출"]`과 `df[["지역", "매출"]]`의 결과 자료구조는 어떻게 다른가?
2. `loc`와 `iloc`의 차이와 슬라이싱 끝값 포함 여부는 무엇인가?
3. 조건 필터링에서 `&`, `|`, `~`는 각각 무엇이며 괄호가 필요한 이유는 무엇인가?
4. `(df["매출"] >= 1000000).sum()`과 `len(df["매출"] >= 1000000)`은 어떻게 다른가?
5. `query()`로 서울 지역의 매출 100만 원 이상인 행을 어떻게 조회하는가?
6. 조건에 맞는 행의 특정 열만 바꾸려면 어떤 코드가 안전한가?
7. `isin()`과 `between()`은 각각 어떤 조회에 사용하는가?
8. 날짜·시간 열에서 2026년 3월 전체를 빠짐없이 조회하려면 종료 조건을 어떻게 지정해야 하는가?
9. 제품명에 특정 단어가 포함된 행을 찾는 메서드는 무엇인가?
10. 매출 상위 5건, 이익 하위 3건을 각각 선택하려면 어떤 메서드를 사용하는가?
11. `sum(axis=0)`과 `sum(axis=1)`의 결과 차이는 무엇인가?


## **확인문제 정답**

### **1. `df["매출"]`과 `df[["지역", "매출"]]`의 결과 자료구조는 어떻게 다른가?**

대괄호 안에 열 이름 하나를 넣으면 **Series(1차원)** 가 되고, 열 이름 목록(리스트)을 넣으면 **DataFrame(2차원 표)** 이 된다.

```python
print(type(df["매출"]))             # Series
print(type(df[["지역", "매출"]]))    # DataFrame

print(df["매출"].shape)             # (15,)    1차원: 숫자가 하나
print(df[["지역", "매출"]].shape)    # (15, 2)  2차원: (행 수, 열 수)
```

열이 하나라도 `df[["매출"]]`처럼 리스트로 넣으면 결과는 DataFrame이다. 모양은 `(15, 1)`이다.

---

### **2. `loc`와 `iloc`의 차이와 슬라이싱 끝값 포함 여부는 무엇인가?**

`loc`는 **인덱스 레이블(이름)과 열 이름**으로 고르고, `iloc`는 **0부터 시작하는 정수 위치**로 고른다.

| 구분 | `loc` | `iloc` |
|---|---|---|
| 기준 | 인덱스 레이블, 열 이름, 조건식 | 정수 위치 |
| 슬라이싱 끝값 | 포함한다 | 포함하지 않는다 |

```python
print(len(df.loc[0:3]))    # 4  레이블 0, 1, 2, 3
print(len(df.iloc[0:3]))   # 3  위치 0, 1, 2
```

---

### **3. 조건 필터링에서 `&`, `|`, `~`는 각각 무엇이며 괄호가 필요한 이유는 무엇인가?**

| 기호 | 뜻 |
|---|---|
| `&` | 그리고: 두 조건을 모두 만족 |
| `\|` | 또는: 둘 중 하나라도 만족 |
| `~` | 반대: 조건을 만족하지 않음 |

`&`, `|`는 `>=`, `==` 같은 비교 기호보다 먼저 계산된다. 괄호가 없으면 계산 순서가 바뀌어 오류가 나므로, 조건 하나하나를 괄호로 묶는다.

```python
df[(df["지역"] == "서울") & (df["매출"] >= 1000000)]   # 서울이면서 100만 이상 → 3행
df[(df["지역"] == "서울") | (df["지역"] == "부산")]     # 서울 또는 부산 → 11행
df[~(df["지역"] == "서울")]                           # 서울이 아닌 행 → 9행

# df[df["지역"] == "서울" & df["매출"] >= 1000000]     # 괄호가 없어 TypeError 발생
```

`and`, `or`, `not`은 쓰지 않는다. 조건식의 결과는 행마다 참·거짓이 담긴 목록이라, 파이썬이 참인지 거짓인지 하나로 판단할 수 없어 `ValueError`가 발생한다.

---

### **4. `(df["매출"] >= 1000000).sum()`과 `len(df["매출"] >= 1000000)`은 어떻게 다른가?**

`df["매출"] >= 1000000`은 행마다 `True`/`False`를 담은 Series이다.

- `sum()`은 `True`를 1, `False`를 0으로 더하므로 **조건을 만족하는 행 수**가 나온다.
- `len()`은 참·거짓과 상관없이 **전체 행 수**를 센다.

```python
print((df["매출"] >= 1000000).sum())   # 6   조건을 만족하는 행 수
print(len(df["매출"] >= 1000000))      # 15  전체 행 수
```

조건에 맞는 행의 개수를 셀 때는 `sum()`을 쓴다.

---

### **5. `query()`로 서울 지역의 매출 100만 원 이상인 행을 어떻게 조회하는가?**

조건을 문자열로 적는다. 열 이름은 따옴표 없이 쓰고, 두 조건은 `and`로 연결한다. 값 `"서울"`은 바깥 따옴표와 다른 따옴표로 감싼다.

```python
df.query('지역 == "서울" and 매출 >= 1000000')   # 노트북, 태블릿, 스마트폰 → 3행

# 조건 필터링으로 쓰면 같은 결과다.
df[(df["지역"] == "서울") & (df["매출"] >= 1000000)]
```

기준 값을 변수에 담았다면 `query()` 안에서 `@변수명`으로 쓴다.

```python
minimum = 1000000
df.query('지역 == "서울" and 매출 >= @minimum')
```

---

### **6. 조건에 맞는 행의 특정 열만 바꾸려면 어떤 코드가 안전한가?**

`copy()`로 복사본을 만든 뒤, `loc[조건, 열] = 값`처럼 **`loc` 한 번에** 행 조건과 열을 함께 지정한다.

```python
graded = df[["주문번호", "매출"]].copy()   # 원본 df를 보호한다
graded["등급"] = "일반"                    # 모든 행에 기본값
graded.loc[graded["매출"] >= 1000000, "등급"] = "고액"   # 조건이 참인 행의 등급만 바꾼다

# 피해야 할 코드: 행을 고른 결과에 다시 열을 골라 값을 넣는다(연속 대입)
# graded[graded["매출"] >= 1000000]["등급"] = "고액"
```

연속 대입은 원본이 바뀌지 않을 수 있고, pandas 버전에 따라 `SettingWithCopyWarning` 같은 경고가 나온다.

---

### **7. `isin()`과 `between()`은 각각 어떤 조회에 사용하는가?**

| 메서드 | 쓰는 상황 | 예 |
|---|---|---|
| `isin()` | 값이 **목록 중 하나**에 해당하는 행 | 지역이 서울 또는 부산 |
| `between()` | 값이 **두 값 사이(범위)** 에 있는 행 | 매출 50만 이상 150만 이하 |

```python
df[df["지역"].isin(["서울", "부산"])]          # 11행
df[~df["지역"].isin(["서울", "부산"])]         # 목록에 없는 지역(대전) → 4행

df[df["매출"].between(500000, 1500000)]       # 50만 이상 150만 이하 → 9행
```

`between()`은 기본으로 양쪽 끝값을 포함한다. 끝값 포함 여부는 `inclusive`(`"both"`, `"left"`, `"right"`, `"neither"`)로 바꾼다.

---

### **8. 날짜·시간 열에서 2026년 3월 전체를 빠짐없이 조회하려면 종료 조건을 어떻게 지정해야 하는가?**

종료 조건을 **다음 달 1일 0시 미만**(`< "2026-04-01"`)으로 지정한다.

`between("2026-03-01", "2026-03-31")`의 종료값은 3월 31일 **0시 0분**이다. 그래서 3월 31일 14시 30분처럼 시간이 붙은 기록은 빠진다.

```python
logs = pd.DataFrame({"주문일": pd.to_datetime([
    "2026-03-01 09:00", "2026-03-31 00:00", "2026-03-31 14:30", "2026-04-01 00:00"])})

# 3월 31일 14:30 기록이 빠진다 → 2건
print(len(logs[logs["주문일"].between("2026-03-01", "2026-03-31")]))

# 3월 전체가 포함된다 → 3건
print(len(logs[(logs["주문일"] >= "2026-03-01") & (logs["주문일"] < "2026-04-01")]))
```

---

### **9. 제품명에 특정 단어가 포함된 행을 찾는 메서드는 무엇인가?**

`str.contains()`를 쓴다. 값이 완전히 같을 때만 참인 `==`와 달리, 글자가 **들어 있기만 하면** 참이다.

```python
df[df["제품명"].str.contains("세트", na=False)]   # 간식세트, 차세트 → 2행
```

| 옵션·메서드 | 뜻 |
|---|---|
| `na=False` | 결측값을 `False`로 처리해 오류 없이 조회한다 |
| `case=False` | 영문 대소문자를 구분하지 않는다 |
| `str.startswith()` | 그 글자로 **시작**하는 값 |
| `str.endswith()` | 그 글자로 **끝나는** 값 |

---

### **10. 매출 상위 5건, 이익 하위 3건을 각각 선택하려면 어떤 메서드를 사용하는가?**

큰 값부터 고를 때는 `nlargest()`, 작은 값부터 고를 때는 `nsmallest()`를 쓴다. 첫 번째 인자는 개수, 두 번째 인자는 기준 열이다.

```python
df.nlargest(5, "매출")    # 매출 상위 5건: 스마트폰, 노트북, 태블릿, 이어폰, 모니터

orders.nsmallest(3, "Profit")[["Product Name", "Profit"]]   # 이익 하위 3건
```

`sort_values("매출", ascending=False).head(5)`와 목적은 같지만 한 번에 쓸 수 있다.

---

### **11. `sum(axis=0)`과 `sum(axis=1)`의 결과 차이는 무엇인가?**

| 지정 | 계산 방향 | 결과 |
|---|---|---|
| `axis=0` | 각 열에서 위에서 아래로(↓) | 열별 합계 |
| `axis=1` | 각 행에서 왼쪽에서 오른쪽으로(→) | 행별 합계 |

```python
axis_ex = pd.DataFrame({"1분기": [100, 200, 300], "2분기": [150, 250, 350]},
                       index=["M01", "M02", "M03"])

print(axis_ex.sum(axis=0))   # 1분기 600, 2분기 750         → 열별 합계
print(axis_ex.sum(axis=1))   # M01 250, M02 450, M03 650    → 행별 합계
```

`axis=0`은 행들을 하나로 합치므로 열마다 결과가 하나씩 남고, `axis=1`은 열들을 하나로 합치므로 행마다 결과가 하나씩 남는다.